In [1]:
import pandas as pd
import sqlite3

# Load the clean data and put it in a SQLite database
df = pd.read_csv("data/clean/reviews_clean.csv")
conn = sqlite3.connect("data/reviews.db")
df.to_sql("reviews", conn, if_exists="replace", index=False)
print("Database ready! Rows:", len(df))

# Small helper: run any SQL query and show the result as a table
def sql(query):
    return pd.read_sql(query, conn)

Database ready! Rows: 46025


In [2]:
sql("""
SELECT app,
       COUNT(*)             AS total_reviews,
       ROUND(AVG(score), 2) AS avg_rating
FROM reviews
GROUP BY app
ORDER BY avg_rating DESC
""")

,app,total_reviews,avg_rating
0,Mrsool,11280,3.26
1,Keeta,12947,3.12
2,HungerStation,13667,2.71
3,ToYou,3312,2.51
4,Jahez,4819,2.29


In [ ]:
import pandas as pd
import pyodbc
import urllib
from sqlalchemy import create_engine
from sqlalchemy.types import NVARCHAR

# Find the SQL Server driver installed on this computer
drivers = [d for d in pyodbc.drivers() if "SQL Server" in d]
print("Drivers found:", drivers)
driver = drivers[-1]

# Connect to our database
params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER=SALEH\\SQLEXPRESS11;DATABASE=DeliveryReviews;"
    "Trusted_Connection=yes;TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

# Load clean data and fix date columns
df = pd.read_csv("data/clean/reviews_clean.csv")
df["at"] = pd.to_datetime(df["at"])
df["repliedAt"] = pd.to_datetime(df["repliedAt"])

# Text columns as NVARCHAR (N = supports Arabic)
text_types = {
    "reviewId": NVARCHAR(100),
    "content": NVARCHAR(1000),
    "clean_text": NVARCHAR(1000),
    "replyContent": NVARCHAR(2000),
    "reviewCreatedVersion": NVARCHAR(50),
    "appVersion": NVARCHAR(50),
    "app": NVARCHAR(50),
    "topics_list": NVARCHAR(300),
}

df.to_sql("reviews", engine, if_exists="replace", index=False,
          dtype=text_types, chunksize=1000)
print("Uploaded to SQL Server! Rows:", len(df))

Drivers found: ['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']
Uploaded to SQL Server! Rows: 46025
